In [3]:
# Cell 1 — Run bandit simulation on Flipkart data
from river import bandit as rb
import pandas as pd, numpy as np, json

ARMS = [0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15]
bandit = rb.UCB(delta=1.0, seed=42)

for arm in ARMS:
    bandit.update(arm, 1.0)

df = pd.read_csv('../data/flipkart_processed.csv')

arm_rewards = {arm: [] for arm in ARMS}

for _, row in df.iterrows():
    arm       = bandit.pull(ARMS)
    sim_price = row['discounted_price'] * arm
    sim_qty   = max(1, 50 * (1 - 0.5 * (arm - 1)))

    base_revenue = row['discounted_price'] * 50
    sim_revenue  = sim_price * sim_qty
    reward_ratio = sim_revenue / (base_revenue + 1e-9)

    bandit.update(arm, reward_ratio)
    arm_rewards[arm].append(sim_revenue)

avg_rewards = {arm: np.mean(vals) if vals else 0 for arm, vals in arm_rewards.items()}
best = max(avg_rewards, key=avg_rewards.get)

arm_stats = {
    str(arm): {
        'avg_reward': float(np.mean(arm_rewards[arm])) if arm_rewards[arm] else 0.0,
        'n_pulls': len(arm_rewards[arm])
    }
    for arm in ARMS
}

bandit_out = {'arms': arm_stats, 'best_arm': str(best)}

with open('../models/bandit_state.json', 'w') as f:
    json.dump(bandit_out, f, indent=2)

print(f"✅ Best multiplier: {best}x")
print(f"✅ Bandit state saved")

✅ Best multiplier: 1.15x
✅ Bandit state saved
